# 34_Video 모델의 학습 수행하기

## 학습목표 
- 1. 저장한 가중치를 불러와 새 데이터에 대한 추론을 수행합니다.

In [5]:
import os
import cv2
import numpy as np
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision import models, transforms
from PIL import Image

In [3]:
class CustomUCF50Dataset(Dataset):
    def __init__(self, root_dir, transform=None, num_frames=16):
        """
        UCF50 비디오 데이터셋을 PyTorch 데이터셋 클래스로 변환.
        
        :param root_dir: UCF50 데이터셋의 루트 디렉토리
        :param transform: 데이터 전처리 및 Augmentation
        :param num_frames: 샘플당 사용할 프레임 개수
        """
        self.root_dir = root_dir
        self.transform = transform
        self.num_frames = num_frames

        # 클래스별 디렉토리를 탐색하여 비디오 파일을 리스트에 저장
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        print(f"CLS : {len(self.classes)}")

        # 모든 비디오 파일의 경로 및 레이블을 리스트에 저장
        self.video_list = []
        for cls in self.classes:
            class_path = os.path.join(root_dir, cls)
            video_files = [f for f in os.listdir(class_path) if f.endswith(('.avi', '.mp4', '.mov', '.mkv'))]

            #print(video_files)
            for video in video_files:
                self.video_list.append((os.path.join(class_path, video), self.class_to_idx[cls]))

        #'C:/Users/jeong/Desktop/OnePM/Projects/Datasets/UCF50/UCF50/BaseballPitch\\v_BaseballPitch_g01_c01.avi'
        #print(self.video_list)

    def __len__(self):
        """ 데이터셋 크기 반환 """
        return len(self.video_list)

    def __getitem__(self, idx):
        """
        비디오 데이터를 읽어 PyTorch Tensor로 변환하여 반환.
        :param idx: 데이터 인덱스
        :return: (프레임 텐서, 레이블)
        """
        video_path, label = self.video_list[idx]

        # 비디오에서 프레임 로드
        frames = self._load_video_frames(video_path, self.num_frames)

        # 변환 적용 (torchvision.transforms 활용)
        if self.transform:
            frames = torch.stack([self.transform(frame) for frame in frames])

        return frames, torch.tensor(label, dtype=torch.long)

    def _load_video_frames(self, video_path, num_frames):
        """
        주어진 비디오에서 num_frames 개의 프레임을 균등한 간격으로 샘플링하여 반환.
        :param video_path: 비디오 파일 경로
        :param num_frames: 가져올 프레임 개수
        :return: [num_frames, H, W, C] 형태의 NumPy 배열 리스트
        """
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total_frames == 0:
            cap.release()
            raise ValueError(f"비디오 {video_path}에서 프레임을 로드할 수 없습니다.")

        # 균등한 간격으로 프레임 샘플링
        frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
        frames = []

        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # OpenCV는 BGR 형식이므로 RGB로 변환
            frame = torch.tensor(frame, dtype=torch.float32) / 255.0  # [H, W, C] 정규화
            frames.append(frame)

        cap.release()

        # 프레임이 부족할 경우 마지막 프레임을 반복하여 채움
        while len(frames) < num_frames:
            frames.append(frames[-1].clone())

        frames = torch.stack(frames, dim=0)  # (num_frames, H, W, C)
        frames = frames.permute(0, 3, 1, 2)  # (num_frames, C, H, W)로 변환

        return frames

In [7]:
def _load_video_frames(video_path, num_frames):
    """
    주어진 비디오에서 num_frames 개의 프레임을 균등한 간격으로 샘플링하여 반환.
    :param video_path: 비디오 파일 경로
    :param num_frames: 가져올 프레임 개수
    :return: [num_frames, H, W, C] 형태의 NumPy 배열 리스트
    """
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        cap.release()
        raise ValueError(f"비디오 {video_path}에서 프레임을 로드할 수 없습니다.")

    # 균등한 간격으로 프레임 샘플링
    frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
    frames = []

    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # OpenCV는 BGR 형식이므로 RGB로 변환
        frame = torch.tensor(frame, dtype=torch.float32) / 255.0  # [H, W, C] 정규화
        frames.append(frame)

    cap.release()

    # 프레임이 부족할 경우 마지막 프레임을 반복하여 채움
    while len(frames) < num_frames:
        frames.append(frames[-1].clone())

    frames = torch.stack(frames, dim=0)  # (num_frames, H, W, C)
    frames = frames.permute(0, 3, 1, 2)  # (num_frames, C, H, W)로 변환

    return frames

#데이터 파일을 불러들임
test_file = 'C:/Users/jeong/Desktop/OnePM/Projects/Datasets/UCF50/UCFTest/v_BaseballPitch_g08_c04.avi'

#파일에 대해 _load_video_frames를 수행
test_frame = _load_video_frames(test_file, 5)

# 데이터 변환 설정 (Resizing + ToTensor)
transform = transforms.Compose([
    transforms.Resize((112, 112)),  # 입력 크기 맞추기
])

# transform.Compose 진행 후 텐서로 변환 
frames = torch.stack([transform(frame) for frame in test_frame])

## 모델 세팅하기

In [9]:
# ✅ MobileNetV3 백본 추출
mobilenet_v3 = models.mobilenet_v3_large(pretrained=True)
mobilenet_v3_backbone = mobilenet_v3.features  # 백본만 사용

class MobileNetFeatureExtractor(nn.Module):
    """ MobileNetV3에서 Feature Map을 추출하여 LSTM 입력 형태로 변환 """
    def __init__(self, backbone, output_dim=512):
        super().__init__()
        self.backbone = backbone  # ✅ MobileNetV3 CNN 백본
        self.global_pool = nn.AdaptiveAvgPool2d(1)  # ✅ Feature Map을 1x1로 축소
        self.fc = nn.Linear(960, output_dim)  # ✅ MobileNetV3 출력 채널(960)을 LSTM 입력 크기(512)로 변환

    def forward(self, x):
        batch_size, num_frames, channels, height, width = x.shape
        x = x.view(batch_size * num_frames, channels, height, width)  # [B*T, C, H, W]

        # ✅ CNN Backbone을 통해 Feature Map 추출
        features = self.backbone(x)  # [B*T, 960, H', W']
        features = self.global_pool(features)  # [B*T, 960, 1, 1]
        features = features.view(batch_size * num_frames, -1)  # [B*T, 960]

        # ✅ FC Layer를 통해 LSTM 입력 크기로 변환
        features = self.fc(features)  # [B*T, 512]
        features = features.view(batch_size, num_frames, -1)  # [B, T, 512]

        return features  # LSTM 입력 형식으로 변환된 Feature Vector

class ActionClassifier(nn.Module):
    def __init__(self, feature_extractor, num_classes, hidden_dim=256, num_layers=2):
        super().__init__()
        self.feature_extractor = feature_extractor  # ✅ MobileNetV3 기반 Feature Extractor
        self.lstm = nn.LSTM(input_size=512, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        features = self.feature_extractor(x)  # ✅ CNN 백본으로 Feature 추출 → [B, T, 512]

        # ✅ LSTM으로 Temporal 정보 학습
        lstm_out, _ = self.lstm(features)  # [B, T, hidden_dim]
        action_logits = self.fc(lstm_out[:, -1, :])  # 마지막 타임스텝의 출력만 사용

        return action_logits  # [B, num_classes]

# ✅ 모델 생성
feature_extractor = MobileNetFeatureExtractor(mobilenet_v3_backbone)
action_classifier = ActionClassifier(feature_extractor, num_classes=15)

C:\ProgramData\anaconda3\envs\tVision\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\ProgramData\anaconda3\envs\tVision\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [11]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"사용 중인 디바이스: {device}")

사용 중인 디바이스: cuda:0


## 추론 수행하기

In [12]:
# 저장된 가중치를 불러와 추론 수행
action_classifier.load_state_dict(torch.load("./video_model.pth", map_location=device))
action_classifier.to(device)
action_classifier.eval()  # 추론 모드

ActionClassifier(
  (feature_extractor): MobileNetFeatureExtractor(
    (backbone): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        (2): Hardswish()
      )
      (1): InvertedResidual(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (1): Conv2dNormActivation(
            (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          )
        )
      )
      (2): InvertedResidual(
        (block): Sequential(
     

1. 데이터준비 -> train데이터셋과 동일한 전처리 수행
2. 모델 준비 
3. 준비한 모델에 가중치 파일 병합
4. 1의 데이터에 배치 차원을 추가
5. 추론 수행

In [ ]:
###!!! 추론할 데이터의 '사이즈'를 맞추는 작업
# [num_frame, C, H, W] -> [1, num_frame, C, H, W]
# 배치사이즈
input_tensor = transform(frames).unsqueeze(0).to(device)

# 추론
with torch.no_grad():
    output = action_classifier(input_tensor)
    predicted_class = output.argmax(dim=1).item()

print(f"✅ 예측된 클래스: {predicted_class}")

✅ 예측된 클래스: 11


In [ ]:
CLS : {'BaseballPitch': 0, 'Biking': 1, 'Diving': 2, 'Fencing': 3, 'HorseRace': 4, 'JumpRope': 5, 'PlayingGuitar': 6, 'PlayingPiano': 7, 'Punch': 8, 'SalsaSpin': 9, 'Skiing': 10, 'Swing': 11, 'TennisSwing': 12, 'VolleyballSpiking': 13, 'YoYo': 14}